In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/playground-series-s5e2/sample_submission.csv
/kaggle/input/playground-series-s5e2/train.csv
/kaggle/input/playground-series-s5e2/test.csv
/kaggle/input/playground-series-s5e2/training_extra.csv


In [2]:
!pip install cmaes

In [3]:
!pip install --upgrade optuna

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 383.6/383.6 kB 6.9 MB/s eta 0:00:00
  Attempting uninstall: optuna
    Found existing installation: optuna 4.2.0
    Uninstalling optuna-4.2.0:
      Successfully uninstalled optuna-4.2.0


In [4]:
!pip install --upgrade scikit-learn==1.4 --no-cache-dir

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.1/12.1 MB 221.3 MB/s eta 0:00:00
  Attempting uninstall: scikit-learn
    Found existing installation: scikit-learn 1.2.2
    Uninstalling scikit-learn-1.2.2:
      Successfully uninstalled scikit-learn-1.2.2


In [5]:
#all import
import pandas as pd
import numpy as np
from functools import partial
from copy import deepcopy
import gc

# Import libraries for gradient boosting
import optuna
from optuna.samplers import CmaEsSampler
from sklearn.base import BaseEstimator, TransformerMixin
import xgboost as xgb
import lightgbm as lgb
# from sklearn.ensemble import RandomForestRegressor, HistGradientBoostingRegressor, GradientBoostingRegressor
# from sklearn.svm import NuSVC, SVC
# from sklearn.neighbors import KNeighborsRegressor
# from sklearn.linear_model import LogisticRegression
# from sklearn.neural_network import MLPRegressor
# from sklearn.gaussian_process import GaussianProcessRegressor
# from sklearn.gaussian_process.kernels import RBF
from catboost import CatBoost, CatBoostRegressor, CatBoostRegressor
from catboost import Pool
from category_encoders import OneHotEncoder, OrdinalEncoder
from sklearn.metrics import root_mean_squared_error, log_loss
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import StratifiedKFold, KFold


In [6]:
#read data 
df_train=pd.read_csv('/kaggle/input/playground-series-s5e2/train.csv')
df_train_xtra=pd.read_csv('/kaggle/input/playground-series-s5e2/training_extra.csv')
df_test=pd.read_csv('/kaggle/input/playground-series-s5e2/test.csv')
df_sub=pd.read_csv('/kaggle/input/playground-series-s5e2/sample_submission.csv')

In [7]:
missing_weight_ids=df_test[df_test['Weight Capacity (kg)'].isna()].id.values

In [8]:
df_train.id.values.max()

299999

In [9]:
#See all columns
df_train.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 300000 entries, 0 to 299999
Data columns (total 11 columns):
 #   Column                Non-Null Count   Dtype  
---  ------                --------------   -----  
 0   id                    300000 non-null  int64  
 1   Brand                 290295 non-null  object 
 2   Material              291653 non-null  object 
 3   Size                  293405 non-null  object 
 4   Compartments          300000 non-null  float64
 5   Laptop Compartment    292556 non-null  object 
 6   Waterproof            292950 non-null  object 
 7   Style                 292030 non-null  object 
 8   Color                 290050 non-null  object 
 9   Weight Capacity (kg)  299862 non-null  float64
 10  Price                 300000 non-null  float64
dtypes: float64(3), int64(1), object(7)
memory usage: 25.2+ MB


In [10]:
df_train_xtra.id.values.max()

4194317

In [11]:
df_train_xtra.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3694318 entries, 0 to 3694317
Data columns (total 11 columns):
 #   Column                Dtype  
---  ------                -----  
 0   id                    int64  
 1   Brand                 object 
 2   Material              object 
 3   Size                  object 
 4   Compartments          float64
 5   Laptop Compartment    object 
 6   Waterproof            object 
 7   Style                 object 
 8   Color                 object 
 9   Weight Capacity (kg)  float64
 10  Price                 float64
dtypes: float64(3), int64(1), object(7)
memory usage: 310.0+ MB


In [12]:
df_train=pd.concat([df_train,df_train_xtra])

In [13]:
df_train.info()

<class 'pandas.core.frame.DataFrame'>
Index: 3994318 entries, 0 to 3694317
Data columns (total 11 columns):
 #   Column                Dtype  
---  ------                -----  
 0   id                    int64  
 1   Brand                 object 
 2   Material              object 
 3   Size                  object 
 4   Compartments          float64
 5   Laptop Compartment    object 
 6   Waterproof            object 
 7   Style                 object 
 8   Color                 object 
 9   Weight Capacity (kg)  float64
 10  Price                 float64
dtypes: float64(3), int64(1), object(7)
memory usage: 365.7+ MB


In [14]:
df_train[df_train['id']>299999]

,id,Brand,Material,Size,Compartments,Laptop Compartment,Waterproof,Style,Color,Weight Capacity (kg),Price
0,500000,Under Armour,Canvas,Small,10.0,Yes,Yes,Tote,Blue,23.882052,114.11068
1,500001,Puma,Polyester,Small,4.0,No,Yes,Backpack,Green,11.869095,129.74972
2,500002,Jansport,Polyester,Small,8.0,Yes,Yes,Tote,Red,8.092302,21.37370
3,500003,Nike,Nylon,Large,7.0,No,No,Messenger,Pink,7.719581,48.09209
4,500004,Nike,Leather,Large,9.0,No,Yes,Tote,Green,22.741826,77.32461
...,...,...,...,...,...,...,...,...,...,...,...
3694313,4194313,Nike,Canvas,NaN,3.0,Yes,Yes,Messenger,Blue,28.098120,104.74460
3694314,4194314,Puma,Leather,Small,10.0,Yes,Yes,Tote,Blue,17.379531,122.39043
3694315,4194315,Jansport,Canvas,Large,10.0,No,No,Backpack,Red,17.037708,148.18470
3694316,4194316,Puma,Canvas,NaN,2.0,No,No,Backpack,Gray,28.783339,22.32269


In [15]:
df_train.isna().any()

id                      False
Brand                    True
Material                 True
Size                     True
Compartments            False
Laptop Compartment       True
Waterproof               True
Style                    True
Color                    True
Weight Capacity (kg)     True
Price                   False
dtype: bool

In [16]:
df_train=df_train.rename(columns={'Laptop Compartment':'Laptop_Compartment','Weight Capacity (kg)':'Weight_Capacity_(kg)'})
df_test=df_test.rename(columns={'Laptop Compartment':'Laptop_Compartment','Weight Capacity (kg)':'Weight_Capacity_(kg)'})

In [17]:
for c in df_train.columns:
    if c not in ['id','Compartments','Price']:
        if df_train[c].dtype==np.number:
            print(f'col {c}\t{df_train[c].min()},{df_train[c].max()},{df_train[c].mean()}')
        else:
            print(f'col {c}\t{df_train[c].unique()}')

<ipython-input-17-8652e0be35bb>:3: DeprecationWarning: Converting `np.inexact` or `np.floating` to a dtype is deprecated. The current result is `float64` which is not strictly correct.
  if df_train[c].dtype==np.number:


col Brand	['Jansport' 'Under Armour' 'Nike' 'Adidas' 'Puma' nan]
col Material	['Leather' 'Canvas' 'Nylon' nan 'Polyester']
col Size	['Medium' 'Small' 'Large' nan]
col Laptop_Compartment	['Yes' 'No' nan]
col Waterproof	['No' 'Yes' nan]
col Style	['Tote' 'Messenger' nan 'Backpack']
col Color	['Black' 'Green' 'Red' 'Blue' 'Gray' 'Pink' nan]
col Weight_Capacity_(kg)	5.0,30.0,18.0104168149865


In [18]:
for c in df_test.columns:
    if c not in ['id','Compartments','Price']:
        if df_test[c].dtype==np.number:
            print(f'col {c}\t{df_test[c].min()},{df_test[c].max()},{df_test[c].mean()}')
        else:
            print(f'col {c}\t{df_test[c].unique()}')

col Brand	['Puma' 'Nike' 'Adidas' nan 'Under Armour' 'Jansport']
col Material	['Leather' 'Canvas' 'Nylon' 'Polyester' nan]
col Size	['Small' 'Medium' 'Large' nan]
col Laptop_Compartment	['No' 'Yes' nan]
col Waterproof	['No' 'Yes' nan]
col Style	['Tote' 'Backpack' 'Messenger' nan]
col Color	['Green' 'Blue' 'Black' 'Red' 'Pink' 'Gray' nan]
col Weight_Capacity_(kg)	5.0,30.0,17.993033231213683


<ipython-input-18-fc51ea396d26>:3: DeprecationWarning: Converting `np.inexact` or `np.floating` to a dtype is deprecated. The current result is `float64` which is not strictly correct.
  if df_test[c].dtype==np.number:


In [19]:
#bag that has no weight values counts
df_train['Weight_Capacity_(kg)'].isna().value_counts()

Weight_Capacity_(kg)
False    3992510
True        1808
Name: count, dtype: int64

In [20]:
#One Hot encoder 
object_columns= df_train.select_dtypes('object').columns.to_list()
encoder = OneHotEncoder(cols=object_columns)
train_encoder = encoder.fit_transform(df_train[object_columns]).astype(int).add_suffix('_ohe')
test_encoder = encoder.transform(df_test[object_columns]).astype(int).add_suffix('_ohe')

In [21]:
#Drop original cols 
df_train=df_train.drop(object_columns,axis=1)
df_train=pd.concat([df_train,train_encoder],axis=1)

df_test=df_test.drop(object_columns,axis=1)
df_test=pd.concat([df_test,test_encoder],axis=1)


In [22]:
df_train=df_train.dropna()
#df_test=df_test.dropna()

In [23]:
df_train=df_train.drop('id',axis=1)
#df_test=df_test.drop('id',axis=1)

In [24]:
sc = StandardScaler()
df_train[['Weight_Capacity_(kg)']] = sc.fit_transform(df_train[['Weight_Capacity_(kg)']])
df_test[['Weight_Capacity_(kg)']] = sc.transform(df_test[['Weight_Capacity_(kg)']])

In [25]:
#Create input and label for predict missing weights
X_train_w=df_train.copy()
X_train_w=X_train_w.dropna()
y_train_w=X_train_w['Weight_Capacity_(kg)']
X_train_w=X_train_w.drop(['Price','Weight_Capacity_(kg)'],axis=1)

X_test_w=df_test[df_test['Weight_Capacity_(kg)'].isna()].drop(['id','Weight_Capacity_(kg)'],axis=1)  #73 row

In [26]:
X_train=df_train
y_train=df_train['Price']
X_test=df_test
X_test=X_test.drop('id',axis=1)

In [27]:
X_train=X_train.drop('Price',axis=1)

In [28]:
X_train.Compartments.value_counts()

Compartments
1.0     423158
4.0     417042
2.0     407873
3.0     406578
7.0     400723
5.0     399305
9.0     398054
10.0    396135
8.0     383065
6.0     360577
Name: count, dtype: int64

In [29]:
X_test.Compartments.value_counts()

Compartments
1.0     21203
4.0     20788
3.0     20366
7.0     20103
2.0     20079
5.0     20049
10.0    19850
9.0     19752
8.0     19433
6.0     18377
Name: count, dtype: int64

In [30]:
X_test.shape

(200000, 34)

# Optuna

In [31]:
class Regressor:
    def __init__(self, early_stopping_rounds,n_estimators=100, device="cpu", random_state=0):
        self.n_estimators = n_estimators
        self.device = device
        self.random_state = random_state
        self.early_stopping_rounds=early_stopping_rounds
        self.models = self._define_model()
        self.len_models = len(self.models)
        
    def _define_model(self):
        
        xgb_params = {
            'n_estimators': self.n_estimators,
            'learning_rate': 0.05,
            'max_depth': 7,
            'subsample': 1.0,
            'colsample_bytree': 1.0,
            'n_jobs': -1,
            'eval_metric': 'rmse',
            'objective': 'reg:squarederror',
            'verbosity': 0,
            'random_state': self.random_state,
            'early_stopping_rounds': self.early_stopping_rounds
        }
        if self.device == 'gpu':
            xgb_params['tree_method'] = 'gpu_hist'
            xgb_params['predictor'] = 'gpu_predictor'
        
        lgb_params = {
            'n_estimators': self.n_estimators,
            'max_depth': 7,
            'learning_rate': 0.05,
            'subsample': 0.20,
            'colsample_bytree': 0.56,
            'reg_alpha': 0.25,
            'reg_lambda': 5e-08,
            'objective': 'regression',
            'metric': 'rmse',
            'boosting_type': 'gbdt',
            'device': self.device,
            'random_state': self.random_state,
            'early_stopping_rounds':self.early_stopping_rounds
        }
        
        cb_params = {
            'iterations': self.n_estimators,
            'depth': 7,
            'learning_rate': 0.1,
            'l2_leaf_reg': 0.7,
            'random_strength': 0.2,
            'max_bin': 200,
            'od_wait': 65,
            'one_hot_max_size': 70,
            'grow_policy': 'Depthwise',
            'bootstrap_type': 'Bayesian',
            'od_type': 'Iter',
            'eval_metric': 'RMSE',
            'loss_function': 'RMSE',
            'task_type': self.device.upper(),
            'random_state': self.random_state
        }
        
        models = {
            'xgb': xgb.XGBRegressor(**xgb_params),
            'lgb': lgb.LGBMRegressor(**lgb_params),
            'cat': CatBoostRegressor(**cb_params),
        }
        
        return models

In [32]:
class OptunaWeights:
    def __init__(self, random_state):
        self.study = None
        self.weights = None
        self.random_state = random_state

    def _objective(self, trial, y_true, y_preds):
        # Define the weights for the predictions from each model
        weights = [trial.suggest_float(f"weight{n}", 0, 1) for n in range(len(y_preds))]

        # Calculate the weighted prediction
        weighted_pred = np.average(np.array(y_preds).T, axis=1, weights=weights)

        # Calculate the AUC score for the weighted prediction
        score = root_mean_squared_error(y_true, weighted_pred)
        return score

    def fit(self, y_true, y_preds, n_trials=1000):
        optuna.logging.set_verbosity(optuna.logging.ERROR)
        sampler = optuna.samplers.CmaEsSampler(seed=self.random_state)
        self.study = optuna.create_study(sampler=sampler, study_name="OptunaWeights", direction='minimize')
        objective_partial = partial(self._objective, y_true=y_true, y_preds=y_preds)
        self.study.optimize(objective_partial, n_trials=n_trials)
        self.weights = [self.study.best_params[f"weight{n}"] for n in range(len(y_preds))]

    def predict(self, y_preds):
        assert self.weights is not None, 'OptunaWeights error, must be fitted before predict'
        weighted_pred = np.average(np.array(y_preds).T, axis=1, weights=self.weights)
        return weighted_pred

    def fit_predict(self, y_true, y_preds, n_trials=1000):
        self.fit(y_true, y_preds, n_trials=n_trials)
        return self.predict(y_preds)
    
    def weights(self):
        return self.weights
    
#Regression doesn't need ThresholdOptimizer
class ThresholdOptimizer:
    def __init__(self, y_true, y_preds, random_state):
        self.y_true = y_true
        self.y_preds = y_preds
        self.random_state = random_state
        self.sampler = optuna.samplers.CmaEsSampler(seed=self.random_state)
        self.study = optuna.create_study(sampler=self.sampler, study_name="OptunaWeights", direction='maximize')
        self.objective_partial = partial(self._objective, y_true=y_true, y_preds=y_preds)
        
    def _objective(self, trial, y_true, y_preds):
        threshold = trial.suggest_float("threshold", 0, 1)
        score = root_mean_squared_error(y_true, np.where(y_preds > threshold, 1, 0))
        return score
        
    def run_optimization(self, n_trials=1000):
        optuna.logging.set_verbosity(optuna.logging.ERROR)
        self.study.optimize(self.objective_partial, n_trials=n_trials)
        
    def get_best_threshold(self):
        return self.study.best_params["threshold"]

# Training

In [33]:
#TRain to find missing weights
n_splits = 5
random_state = 42
n_estimators = 5000 # 9999
early_stopping_rounds = 50
verbose = False
device = 'gpu'

# Initialize an array for storing test predictions
test_predss = np.zeros(X_test.shape[0])

test_predss_sub = []
ensemble_score = []
weights = []
regressor=Regressor(early_stopping_rounds=early_stopping_rounds)
trained_models = dict(zip(regressor.models.keys(), [[] for _ in range(regressor.len_models)]))

stratify_column=X_train['Compartments']

kf = StratifiedKFold(n_splits=n_splits, random_state=random_state, shuffle=True)
for i, (train_index, val_index) in enumerate(kf.split(X_train_w, stratify_column)):
    X_train_, X_val = X_train_w.iloc[train_index], X_train_w.iloc[val_index]
    y_train_, y_val = y_train_w.iloc[train_index], y_train_w.iloc[val_index]
        
    # Get a set of Regressor models
    rs = Regressor(early_stopping_rounds,n_estimators, device, random_state)
    models = rs.models
    
    # Initialize lists to store oof and test predictions for each base model
    oof_preds = []
    test_preds = []
    
    # Loop over each base model and fit it to the training data, evaluate on validation data, and store predictions
    for name, model in models.items():
        if name == 'cat_label':
            train_pool = Pool(X_train_, y_train_)#, cat_features=categorical_columns)
            test_pool = Pool(X_val, y_val)# ,cat_features=categorical_columns)
            model.fit(train_pool, eval_set=[test_pool], early_stopping_rounds=early_stopping_rounds, verbose=verbose)
        elif name == 'lgb_label':
            model.fit(X_train_, y_train_, eval_set=[(X_val, y_val)])
        elif name in ['xgb', 'lgb', 'cat']:
            model.fit(X_train_, y_train_, eval_set=[(X_val, y_val)])
        else:
            model.fit(X_train_fillna.iloc[train_index], y_train_)
        
        
        # if name in ['xgb', 'lgb', 'cat', 'cat_label', 'lgb_label']:
        #     test_pred = model.predict_proba(X_test)[:, 1]
        #     y_val_pred = model.predict_proba(X_val)[:, 1]
        # else:
        #     test_pred = model.predict_proba(X_test_fillna)[:, 1]
        #     y_val_pred = model.predict_proba(X_train_fillna.iloc[val_index])[:, 1]
        
        test_pred=model.predict(X_test_w)
        y_val_pred=model.predict(X_val)
        score = root_mean_squared_error(y_val, y_val_pred)
        print(f'{name} model [FOLD-{i}] RMSE: {score:.5f}')
        
        oof_preds.append(y_val_pred)
        test_preds.append(test_pred)
        trained_models[f'{name}'].append(deepcopy(model))
    
    # Use Optuna to find the best ensemble weights
    optweights = OptunaWeights(random_state=random_state)
    y_val_pred = optweights.fit_predict(y_val.values, oof_preds)

    #****** Not use Threshold because we do regression *****
    # Use Optuna to find the best threshold
    # opt = ThresholdOptimizer(y_val, y_val_pred, random_state)
    # opt.run_optimization(n_trials=2000)
    # best_threshold = opt.get_best_threshold()
    
    #score = accuracy_score(y_val, np.where(y_val_pred > best_threshold, 1, 0)) 
    score = root_mean_squared_error(y_val, y_val_pred)
    print(f'Ensemble [FOLD-{i}] RMSE score {score:.5f}')
    ensemble_score.append(score)
    weights.append(optweights.weights)
    
    test_predss_sub.append(optweights.predict(test_preds))
    print(f'result added {len(test_predss_sub)} items')
    #test_predss_onehot.append(np.where(optweights.predict(test_preds) > best_threshold, 1, 0))
    #gc.collect()

[0]	validation_0-rmse:0.99946
[1]	validation_0-rmse:0.99943
[2]	validation_0-rmse:0.99941
[3]	validation_0-rmse:0.99939
[4]	validation_0-rmse:0.99938
[5]	validation_0-rmse:0.99936
[6]	validation_0-rmse:0.99935
[7]	validation_0-rmse:0.99933
[8]	validation_0-rmse:0.99932
[9]	validation_0-rmse:0.99931
[10]	validation_0-rmse:0.99930
[11]	validation_0-rmse:0.99929
[12]	validation_0-rmse:0.99929
[13]	validation_0-rmse:0.99928
[14]	validation_0-rmse:0.99927
[15]	validation_0-rmse:0.99927
[16]	validation_0-rmse:0.99926
[17]	validation_0-rmse:0.99926
[18]	validation_0-rmse:0.99925
[19]	validation_0-rmse:0.99925
[20]	validation_0-rmse:0.99925
[21]	validation_0-rmse:0.99924
[22]	validation_0-rmse:0.99924
[23]	validation_0-rmse:0.99923
[24]	validation_0-rmse:0.99923
[25]	validation_0-rmse:0.99923
[26]	validation_0-rmse:0.99922
[27]	validation_0-rmse:0.99922
[28]	validation_0-rmse:0.99922
[29]	validation_0-rmse:0.99922
[30]	validation_0-rmse:0.99921
[31]	validation_0-rmse:0.99921
[32]	validation_0-

In [34]:
#The missing weight capacity
np.median(test_predss_sub, axis=0).reshape(-1,1).flatten()

array([ 0.06741025, -0.12791681,  0.09734696,  0.35094072, -0.03259109,
        0.03264037,  0.42217257,  0.02400568,  0.35894367,  0.38577381,
        0.01697508,  0.05829769, -0.02178566,  0.05426878,  0.03740318,
        0.03698445,  0.03112688,  0.05304039,  0.03832425,  0.04114402,
        0.05271335, -0.01616696, -0.03062218,  0.13107824,  0.04035122,
       -0.0532125 ,  0.0078274 ,  0.13743226, -0.01072255,  0.17255062,
       -0.03546497, -0.06388955,  0.00176594,  0.00258538, -0.00256121,
        0.03510546,  0.00672631,  0.06719677,  0.07195068,  0.07994927,
        0.37856847, -0.10088725,  0.05550055,  0.02252712,  0.00951064,
       -0.03248443,  0.07067256,  0.02059611, -0.01079954,  0.04392741,
        0.06128727,  0.00557696,  0.06848532,  0.07289732,  0.07374648,
        0.01064964,  0.02058298,  0.01530033, -0.03214861,  0.05709394,
        0.04496775,  0.04245212,  0.3519916 , -0.11576178,  0.04220285,
        0.05533076,  0.01027735, -0.08976909,  0.0539236 ,  0.45

In [35]:
X_test.isna().any()

Compartments                False
Weight_Capacity_(kg)         True
Brand_1_ohe                 False
Brand_2_ohe                 False
Brand_3_ohe                 False
Brand_4_ohe                 False
Brand_5_ohe                 False
Brand_6_ohe                 False
Material_1_ohe              False
Material_2_ohe              False
Material_3_ohe              False
Material_4_ohe              False
Material_5_ohe              False
Size_1_ohe                  False
Size_2_ohe                  False
Size_3_ohe                  False
Size_4_ohe                  False
Laptop_Compartment_1_ohe    False
Laptop_Compartment_2_ohe    False
Laptop_Compartment_3_ohe    False
Waterproof_1_ohe            False
Waterproof_2_ohe            False
Waterproof_3_ohe            False
Style_1_ohe                 False
Style_2_ohe                 False
Style_3_ohe                 False
Style_4_ohe                 False
Color_1_ohe                 False
Color_2_ohe                 False
Color_3_ohe   

In [36]:
df_test.loc[df_test.id.isin(missing_weight_ids),['Weight_Capacity_(kg)']] = np.median(test_predss_sub, axis=0).reshape(-1,1).flatten()
df_test[df_test.id.isin(missing_weight_ids)]['Weight_Capacity_(kg)']

1350      0.067410
1895     -0.127917
4609      0.097347
7785      0.350941
9500     -0.032591
            ...   
184623   -0.165147
187325    0.086537
188370   -0.006694
189704    0.400364
191856    0.022023
Name: Weight_Capacity_(kg), Length: 77, dtype: float64

In [37]:
X_test.isna().any()

Compartments                False
Weight_Capacity_(kg)         True
Brand_1_ohe                 False
Brand_2_ohe                 False
Brand_3_ohe                 False
Brand_4_ohe                 False
Brand_5_ohe                 False
Brand_6_ohe                 False
Material_1_ohe              False
Material_2_ohe              False
Material_3_ohe              False
Material_4_ohe              False
Material_5_ohe              False
Size_1_ohe                  False
Size_2_ohe                  False
Size_3_ohe                  False
Size_4_ohe                  False
Laptop_Compartment_1_ohe    False
Laptop_Compartment_2_ohe    False
Laptop_Compartment_3_ohe    False
Waterproof_1_ohe            False
Waterproof_2_ohe            False
Waterproof_3_ohe            False
Style_1_ohe                 False
Style_2_ohe                 False
Style_3_ohe                 False
Style_4_ohe                 False
Color_1_ohe                 False
Color_2_ohe                 False
Color_3_ohe   

# After getting missing weights

In [38]:
n_splits = 10
random_state = 42
n_estimators = 9000 # 9999
early_stopping_rounds = 100
verbose = False
device = 'gpu'

# Initialize an array for storing test predictions
test_predss = np.zeros(X_test.shape[0])

test_predss_sub = []
ensemble_score = []
weights = []
regressor=Regressor(early_stopping_rounds=early_stopping_rounds)
trained_models = dict(zip(regressor.models.keys(), [[] for _ in range(regressor.len_models)]))

stratify_column=X_train['Compartments']

kf = StratifiedKFold(n_splits=n_splits, random_state=random_state, shuffle=True)
for i, (train_index, val_index) in enumerate(kf.split(X_train, stratify_column)):
    X_train_, X_val = X_train.iloc[train_index], X_train.iloc[val_index]
    y_train_, y_val = y_train.iloc[train_index], y_train.iloc[val_index]
        
    # Get a set of Regressor models
    rs = Regressor(early_stopping_rounds,n_estimators, device, random_state)
    models = rs.models
    
    # Initialize lists to store oof and test predictions for each base model
    oof_preds = []
    test_preds = []
    
    # Loop over each base model and fit it to the training data, evaluate on validation data, and store predictions
    for name, model in models.items():
        if name == 'cat_label':
            train_pool = Pool(X_train_, y_train_)#, cat_features=categorical_columns)
            test_pool = Pool(X_val, y_val)# ,cat_features=categorical_columns)
            model.fit(train_pool, eval_set=[test_pool], early_stopping_rounds=early_stopping_rounds, verbose=verbose)
        elif name == 'lgb_label':
            model.fit(X_train_, y_train_, eval_set=[(X_val, y_val)])
        elif name in ['xgb', 'lgb', 'cat']:
            model.fit(X_train_, y_train_, eval_set=[(X_val, y_val)])
        else:
            model.fit(X_train_fillna.iloc[train_index], y_train_)
        
        
        # if name in ['xgb', 'lgb', 'cat', 'cat_label', 'lgb_label']:
        #     test_pred = model.predict_proba(X_test)[:, 1]
        #     y_val_pred = model.predict_proba(X_val)[:, 1]
        # else:
        #     test_pred = model.predict_proba(X_test_fillna)[:, 1]
        #     y_val_pred = model.predict_proba(X_train_fillna.iloc[val_index])[:, 1]
        
        test_pred=model.predict(X_test)
        y_val_pred=model.predict(X_val)
        score = root_mean_squared_error(y_val, y_val_pred)
        print(f'{name} model [FOLD-{i}] RMSE: {score:.5f}')
        
        oof_preds.append(y_val_pred)
        test_preds.append(test_pred)
        trained_models[f'{name}'].append(deepcopy(model))
    
    # Use Optuna to find the best ensemble weights
    optweights = OptunaWeights(random_state=random_state)
    y_val_pred = optweights.fit_predict(y_val.values, oof_preds)

    #****** Not use Threshold because we do regression *****
    # Use Optuna to find the best threshold
    # opt = ThresholdOptimizer(y_val, y_val_pred, random_state)
    # opt.run_optimization(n_trials=2000)
    # best_threshold = opt.get_best_threshold()
    
    #score = accuracy_score(y_val, np.where(y_val_pred > best_threshold, 1, 0)) 
    score = root_mean_squared_error(y_val, y_val_pred)
    print(f'Ensemble [FOLD-{i}] RMSE score {score:.5f}')
    ensemble_score.append(score)
    weights.append(optweights.weights)
    
    test_predss_sub.append(optweights.predict(test_preds))
    print(f'result added {len(test_predss_sub)} items')
    #test_predss_onehot.append(np.where(optweights.predict(test_preds) > best_threshold, 1, 0))
    #gc.collect()

[0]	validation_0-rmse:38.91353
[1]	validation_0-rmse:38.91011
[2]	validation_0-rmse:38.90693
[3]	validation_0-rmse:38.90416
[4]	validation_0-rmse:38.90161
[5]	validation_0-rmse:38.89939
[6]	validation_0-rmse:38.89730
[7]	validation_0-rmse:38.89533
[8]	validation_0-rmse:38.89350
[9]	validation_0-rmse:38.89174
[10]	validation_0-rmse:38.88996
[11]	validation_0-rmse:38.88857
[12]	validation_0-rmse:38.88722
[13]	validation_0-rmse:38.88593
[14]	validation_0-rmse:38.88453
[15]	validation_0-rmse:38.88334
[16]	validation_0-rmse:38.88216
[17]	validation_0-rmse:38.88090
[18]	validation_0-rmse:38.87995
[19]	validation_0-rmse:38.87901
[20]	validation_0-rmse:38.87825
[21]	validation_0-rmse:38.87729
[22]	validation_0-rmse:38.87644
[23]	validation_0-rmse:38.87577
[24]	validation_0-rmse:38.87508
[25]	validation_0-rmse:38.87433
[26]	validation_0-rmse:38.87375
[27]	validation_0-rmse:38.87334
[28]	validation_0-rmse:38.87268
[29]	validation_0-rmse:38.87195
[30]	validation_0-rmse:38.87142
[31]	validation_0-

In [39]:
# import numpy as np
# import pandas as pd

# E=[]
# a=np.array([1.,1.,1.])
# b=np.array([2.,2.,2.])

# c=np.array([a,b])
# d=c.T
# print(np.average(d,axis=1,weights=[[1,2],[1,2],[1,2]]))

# E.append(a)
# E.append(b)
# F=np.median(E,axis=0)
# print(F)

# Evaluation

In [40]:
# Calculate the mean LogLoss score of the ensemble
mean_score = np.mean(ensemble_score)
std_score = np.std(ensemble_score)
print(f'Ensemble Accuracy score {mean_score:.5f} ± {std_score:.5f}')

# Print the mean and standard deviation of the ensemble weights for each model
print('--- Model Weights ---')
mean_weights = np.mean(weights, axis=0)
std_weights = np.std(weights, axis=0)
for name, mean_weight, std_weight in zip(models.keys(), mean_weights, std_weights):
    print(f'{name} {mean_weight:.5f} ± {std_weight:.5f}')

Ensemble Accuracy score 38.86677 ± 0.02650
--- Model Weights ---
xgb 0.18247 ± 0.10041
lgb 0.67286 ± 0.13095
cat 0.24587 ± 0.18096


# Submission# 

In [41]:
df_sub['Price'] = np.median(test_predss_sub, axis=0)
df_sub.to_csv('submission.csv',index=False)
df_sub.head(5)

,id,Price
0,300000,80.911542
1,300001,82.572984
2,300002,82.902290
3,300003,80.892026
4,300004,79.255460
